# Composable CDP — walkthrough

A guided read of the pipeline output. Nothing here builds anything: run `./run.sh`
first, then work down this notebook.

It is ordered as a narrative rather than as a pipeline. The scorecard comes early,
on purpose — if the numbers are bad, that is the conversation, and it is better to
have it on slide two than slide twenty.

In [ ]:
import os, re, pathlib

# Read config.env rather than hardcoding anything. The repository never
# contains a project ID; setup.sh wrote one here and this file is git-ignored.
cfg_path = pathlib.Path('config.env')
assert cfg_path.exists(), 'No config.env — run ./setup.sh first.'

CFG = {}
for line in cfg_path.read_text().splitlines():
    m = re.match(r'^([A-Z_]+)="?(.*?)"?$', line.strip())
    if m and not line.startswith('#'):
        CFG[m.group(1)] = m.group(2)

PROJECT = CFG['CDP_PROJECT']
DS      = CFG['CDP_DS']
DS_T    = CFG['CDP_DS_TRUTH']
LOC     = CFG['CDP_LOCATION']

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location=LOC)

def q(sql):
    """Run a query with {ds} and {truth} substituted, return a DataFrame."""
    return client.query(sql.format(ds=f'{PROJECT}.{DS}', truth=f'{PROJECT}.{DS_T}')).to_dataframe()

print(f'{PROJECT} · {DS} · {LOC}')

## 1 · The problem, before any cleverness

Eight sources. Four of them are missing most of the fields you would want to match
on. This is the honest starting position, and it is the reason a rules-only
approach caps out.

In [ ]:
q("SELECT * FROM `{ds}.v_source_profile`")

## 2 · The scorecard

Measured on this run, on this corpus, with the configuration shown alongside.

Read `clusters_containing_multiple_people` before anything else. Over-merges and
under-merges are not interchangeable: one is a privacy incident, the other is a
data-quality ticket.

In [ ]:
q("SELECT * FROM `{ds}.v_scorecard`").T

## 3 · The hard cases

One row per planted trap, pass or fail, with a diagnosis of *where* each failure
happened — retrieval, scoring, or judgement. They need completely different fixes.

Expect failures. A run where everything passes means the cases are too easy.

In [ ]:
cases = q("SELECT * FROM `{ds}.v_case_results` ORDER BY passed, case_type")
cases[['case_type', 'expected_outcome', 'tp', 'fp', 'fn', 'precision', 'recall', 'passed', 'diagnosis']]

## 4 · Does hybrid retrieval earn its place?

The deck claims semantic-only and lexical-only architectures each miss a distinct
class of true match. Here is that claim tested against ground truth.

If both single-leg recall figures sit close to the hybrid one, hybrid is buying
complexity for very little — and the deck should be changed, not the query.

In [ ]:
q("SELECT * FROM `{ds}.v_retrieval_recall`").T

### Live, single query

The same probe, two retrieval modes. An account number is a rare alphanumeric token:
decisive for BM25, meaningless to an embedding model.

Pick a real account number from the corpus first.

In [ ]:
probe = q("""
  SELECT account_number
  FROM `{ds}.node_account`
  WHERE record_count > 1
  ORDER BY record_count DESC, account_number
  LIMIT 1
""").iloc[0, 0]
print('probe:', probe)

hybrid = q(f"SELECT * FROM `{{ds}}.tf_lookup_hybrid`('{probe}')")
vector = q(f"SELECT * FROM `{{ds}}.tf_lookup_vector`('{probe}')")
print(f'hybrid returned {len(hybrid)}, vector-only returned {len(vector)}')
hybrid.head(10)

## 5 · The funnel

The cost argument in one table. Only the grey zone pays for an LLM.

In [ ]:
display(q("SELECT * FROM `{ds}.v_blocking_funnel`").T)
display(q("SELECT * FROM `{ds}.v_candidate_funnel`"))
display(q("SELECT * FROM `{ds}.v_adjudication_summary`"))

## 6 · The prompt-injection guard

Three ticket bodies contain text instructing the reader to confirm a match. The
adjudicator prompt declares record content to be untrusted data.

If any of these came back as a confident `MATCH`, say so out loud. It is the single
most important thing in the run.

In [ ]:
q("SELECT * FROM `{ds}.v_injection_guard` ORDER BY record_id_a, record_id_b LIMIT 10")

## 7 · One person, end to end

Every field, the value that survived, where it came from, and what it beat.

This is the query that answers "why does the system think this?" — and being able to
answer that is most of what separates a platform from a black box.

In [ ]:
# Pick someone interesting: the most contested profile in the corpus.
pid = q("""
  SELECT person_id
  FROM `{ds}.golden_person`
  ORDER BY contested_fields DESC, populated_fields DESC, person_id
  LIMIT 1
""").iloc[0, 0]
print('person:', pid)

q(f"SELECT * FROM `{{ds}}.tf_explain_person`('{pid}')")

## 8 · Consent

Merging people is not merging permissions.

`contacts_the_union_rule_would_add` is the number for the CMO. Read the other way,
it is the number of unlawful contacts the convenient implementation would send.

In [ ]:
display(q("SELECT * FROM `{ds}.v_consent_impact`"))
display(q("SELECT person_id, channel, granted_count, withdrawn_count, not_given_count, would_have_been_contacted, actual_decision, explanation FROM `{ds}.v_consent_conflicts` LIMIT 5"))

## 9 · The graph, and what it makes possible

Seven node types, eight edge types — then the things that were impossible before.

In [ ]:
display(q("SELECT * FROM `{ds}.v_graph_summary`"))
display(q("SELECT * FROM `{ds}.v_cluster_sizes` LIMIT 15"))

In [ ]:
# Segmentation on people rather than records — and the before/after that shows
# why that distinction is not academic.
display(q("SELECT segment_id, people, avg_recency_days, avg_frequency, avg_spend_aud, g.segment_name, g.suggested_action FROM `{ds}.segment_labels`"))
display(q("SELECT * FROM `{ds}.v_activation_before_after`"))

In [ ]:
# Households, and the duplicate mailings a resolved view avoids.
display(q("SELECT * FROM `{ds}.v_household_waste`").T)

# Shared identifiers — a one-hop question in a graph, a query nobody writes
# against a flat customer table.
display(q("SELECT * FROM `{ds}.v_shared_identifiers` WHERE assessment LIKE 'REVIEW%' ORDER BY people DESC, identifier_type, identifier LIMIT 10"))

## 10 · Grounding an agent

One profile, with provenance and — critically — with its gaps stated. An agent told
that a value is contested will ask. An agent handed a single clean value will assert.

In [ ]:
q(f"""
  SELECT response.answer, response.grounded_in, response.confident
  FROM `{{ds}}.tf_ask_about_person`(
    '{pid}',
    'Can we email this customer a marketing offer, and what is their current address?'
  )
""")

## 11 · What it cost, and what it would cost at your volume

Measured, not estimated — the token counts below are what the models themselves
reported, and the BigQuery figures are what BigQuery actually billed.

Three caveats to say out loud rather than bury:

- **Unit prices are list prices captured 14 Sep 2026.** They live in `config.env`.
  Re-verify before quoting anyone.
- **`AI.EMBED` and `AI.CLASSIFY` report no usage metadata**, so those two components
  are estimated from input length. The `basis` column marks which is which.
- **The extrapolation is linear.** Adjudication volume is the component least likely
  to scale linearly, because blocking discriminates less well as the corpus densifies.
  Treat the target figure as a floor, not a forecast.

In [ ]:
# Per-component consumption, with the honesty column first.
display(q("""
  SELECT basis, component, calls, input_tokens, output_tokens, usd_model_cost
  FROM `{ds}.cost_components`
  ORDER BY usd_model_cost DESC
"""))

# The headline: unit economics and the extrapolation.
display(q("SELECT * FROM `{ds}.v_cost_model`").T)

# BigQuery compute is priced separately and both models are shown, because
# which one applies depends on how the customer buys, not on this workload.
try:
    display(q("SELECT * FROM `{ds}.v_cost_compute`").T)
except Exception as e:
    print("v_cost_compute unavailable — INFORMATION_SCHEMA needs job-listing "
          "permission on the project. Model costs above are unaffected.")
    print(f"  {type(e).__name__}: {e}")

# The lever a customer will actually want to pull. Cheaper is always available;
# what it costs is recall, and v_scorecard is where that shows up.
display(q("SELECT * FROM `{ds}.v_cost_sensitivity`"))

## 12 · Show me one it got wrong

Offer this before being asked. To a technical audience it is worth more than any
headline number.

False positives first — they are the ones that matter.

In [ ]:
q("""
  SELECT outcome, record_a, record_b, case_a, retrieved_by, tier, decision,
         combined_score, dob_conflict, llm_rationale
  FROM `{ds}.v_failures`
  LIMIT 20
""")